In [ ]:
%matplotlib ipympl

import functools
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scipy.signal as sci_sig
import scipy.optimize as sci_opt
import control as ct

import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.comp as comp

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    """Load reference linear accelerations and angular velocities."""
    df = pd.read_csv(file_path)

    acc_keys = [f"sesmt.md.merged_frame.xyz_acc[{i}] {{m/s^2}}" for i in range(3)]
    omega_keys = [f"sesmt.md.merged_frame.ang_vel[{i}] {{rad/s}}" for i in range(3)]
    gravity_keys = [
        f"sesmt.md.merged_frame.gravity[{i}] {{m/s^2}}" for i in range(3)
    ]

    acc_ref = jnp.array(df[acc_keys])
    omega_ref = jnp.array(df[omega_keys])
    gravity_ref = jnp.array(df[gravity_keys])

    # for some reason, data collection after a lot of nonsense data
    # we grab the data after we start recognizing nonzero (x) accelerations
    # note that using direct equality is desired here (and not jnp.isclose)
    offset = jnp.argmax(acc_ref[:, 0] != 0.0)

    # we need to cancel and then add back in the gravity vector
    acc_ref = acc_ref[offset:, :] - 2 * gravity_ref[offset:, :]
    omega_ref = omega_ref[offset:, :]
    
    return acc_ref, omega_ref

def load_specific_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    df = pd.read_csv(file_path)

    ks = df.keys()

    ts = np.array(df[ks[0]])
    diff = np.abs(np.diff(ts))
    avg_diff = np.mean(diff)
    std_diff = np.std(diff)
    if std_diff > 0.05:
        bad_indices = np.where(diff > avg_diff + std_diff)[0] + 1 # off by one
        start_index = bad_indices[-2] + 5 * 200
        end_index = bad_indices[-1] - 1
    else:
        start_index = 0
        end_index = ts.size - 1
    print(f"(start_index, end_index) = ({start_index}, {end_index})")

    # ts = ts[start_index: end_index + 1]
    df = df[start_index: end_index + 1]

    acc_ref = jnp.transpose(jnp.array([df[k] for k in ks[1:4]]))
    omega_ref = jnp.transpose(jnp.array([df[k] for k in ks[4:]]))
    return acc_ref, omega_ref

# file_path = "/Users/jozbee/work/eng/comp/data/00_sms_drive.csv"
# acc_ref, omega_ref = load_sms_references(file_path)

# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander-sim.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-rover-sim.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-standard-road-v2.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander_motion_approach_auto.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander_motion_redes_auto.csv"
file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander_motion_redes_manual.csv"
acc_ref, omega_ref = load_specific_sms_references(file_path)

# data_size = 100 * 200
data_size = acc_ref.shape[0]
acc_ref = np.array(acc_ref)[:data_size]
omega_ref = np.array(omega_ref)[:data_size]

In [ ]:
fs = 0.005
sos = sci_sig.butter(N=20, Wn=fs / 100.0, btype="lowpass", analog=False, output="sos", fs=fs)
butter_aref = sci_sig.sosfilt(sos, acc_ref, axis=0)
butter_oref = sci_sig.sosfilt(sos, omega_ref, axis=0)

In [ ]:
wl = 200
po = 4
savgol_aref0 = sci_sig.savgol_filter(acc_ref, window_length=wl, polyorder=po, axis=0)
savgol_oref0 = sci_sig.savgol_filter(omega_ref, window_length=wl, polyorder=po, axis=0)

wl = 100
po = 0
savgol_aref = sci_sig.savgol_filter(savgol_aref0, window_length=wl, polyorder=po, axis=0)
savgol_oref = sci_sig.savgol_filter(savgol_oref0, window_length=wl, polyorder=po, axis=0)

In [ ]:
@jax.jit
def ab_filt(a, b, dt, x):
    
    def scan_body(xv0, x_obs):
        x0, v0 = xv0
        x1 = x0 + dt * v0
        v1 = v0
        r1 = x_obs - x1
        x1 = x1 + a * r1
        v1 = v1 + b / dt * r1
        xv1 = jnp.array([x1, v1])
        return xv1, xv1

    _, res = jax.lax.scan(scan_body, init=jnp.array([x[0], 0.0]), xs=x)
    return res[:, 0]  # ignore the velocity component

def ab_filt_np(a, b, dt, x, axis=1):
    vmap_ab_filt = jax.vmap(ab_filt, in_axes=[None, None, None, axis])
    return np.transpose(np.array(vmap_ab_filt(a, b, dt, x)))

In [ ]:
vmap_ab_filt = jax.jit(jax.vmap(ab_filt, in_axes=[None, None, None, 1]))
# vmap_ab_filt = jax.jit(jax.vmap(ab_filt, in_axes=[0, 0, None, 1]))

@jax.jit
def ab_err(x, y, dt, ab):
    # ab = ab.reshape(6, 2)
    a = ab[..., 0] * 1e-2
    b = ab[..., 1] * 1e-6
    ab_y = jnp.transpose(vmap_ab_filt(a, b, dt, x))
    return jnp.mean(jnp.square(ab_y - y))

x = jnp.hstack([acc_ref, omega_ref])
y = jnp.hstack([savgol_aref, savgol_oref])
ab_err_fun = jax.value_and_grad(ab_err, argnums=3)  # jac == True

def sci_ab_fun(ab):
    return [np.array(arr) for arr in ab_err_fun(x, y, 0.005, ab)]

ab_res = sci_opt.minimize(
    fun=sci_ab_fun,
    x0=np.ones(2),
    # x0=np.ones(shape=(12,)),
    method="L-BFGS-B",
    jac=True,
    options={"ftol": 1e-16, "gtol": 1e-16},
)
ab_res

In [ ]:
one = sci_ab_fun(np.array([ab_res.x[0], ab_res.x[1]]))
two = sci_ab_fun(np.array([ab_res.x[0], 0.0]))
three = sci_ab_fun(np.array([2., 0.0]))
one, two, three

In [ ]:
# a = 0.01
# b = 0.00001
a = ab_res.x[0] * 1e-2
b = ab_res.x[1] * 1e-6
# b = 0.0
ab_aref = ab_filt_np(a, b, 0.005, acc_ref)
ab_oref = ab_filt_np(a, b, 0.005, omega_ref)

In [ ]:
frac = 0.5098245285339587 / (2 * np.pi * 0.5)
s = ct.tf("s")
sms_tf_ct = 1 / (1  + frac * s)**3
sms_vspec = const.VSpec.transfer2vspec(sms_tf_ct, dt=0.005)

def sms_lti_int(u):
    return comp.lti_int(
        E0=sms_vspec.E0,
        E1=sms_vspec.E1,
        C=sms_vspec.C,
        D=sms_vspec.D,
        x0=np.zeros(sms_vspec.E0.shape[0]),
        u=u,
    )[1]

vmap_sms_lit_int = jax.jit(lambda u: jax.vmap(sms_lti_int, in_axes=1)(u).T)

sms_aref_check = vmap_sms_lit_int(acc_ref)
sms_oref_check = vmap_sms_lit_int(omega_ref)

In [ ]:
# sms filt
z = np.array([])
p = np.ones(3) * -frac**-1
k = 1.0

#######
# BAD #
#######

# sms_tf = sci_sig.zpk2tf(z, p, k)
# sms_tf = (sms_tf[0], sms_tf[1] / sms_tf[1][-1])
# sms_bi = sci_sig.bilinear(*sms_tf, fs=200.0)
# sms_sos = sci_sig.tf2sos(sms_bi[0], sms_bi[1])

# sms_bi = sci_sig.bilinear_zpk(z, p, k, fs=200.0)
# sms_sos = sci_sig.zpk2sos(sms_bi[0], sms_bi[1], sms_bi[2])

# sms_bi = sci_sig.cont2discrete((z, p, k), 0.005, method="bilinear")
# sms_sos = sci_sig.zpk2sos(sms_bi[0], sms_bi[1], sms_bi[2])

########
# GOOD #
########

# sms_tf_disc = sms_tf_ct.sample(Ts=0.005, method="bilinear")
# sms_sos = sci_sig.zpk2sos(sms_tf_disc.zeros(), sms_tf_disc.poles(), -sms_tf_disc(0.0))

sms_tf = sci_sig.zpk2tf(z, p, k)
sms_tf = (sms_tf[0], sms_tf[1] / sms_tf[1][-1])
sms_bi = sci_sig.cont2discrete(sms_tf, 0.005, method="bilinear")
sms_sos = sci_sig.tf2sos(sms_bi[0][0, :], sms_bi[1])

sms_aref = sci_sig.sosfilt(sms_sos, acc_ref, axis=0)
sms_oref = sci_sig.sosfilt(sms_sos, omega_ref, axis=0)

In [ ]:
plt.close("all")

In [ ]:
# filt_aref = butter_aref
# filt_oref = butter_oref

# filt_aref = savgol_aref0
# filt_oref = savgol_oref0

# filt_aref = savgol_aref
# filt_oref = savgol_oref

# filt_aref = ab_aref
# filt_oref = ab_oref

# filt_aref = sms_aref_check
# filt_oref = sms_oref_check

filt_aref = sms_aref
filt_oref = sms_oref

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 8), sharex=True)

axs[0, 0].plot(acc_ref[:, 0], label="acc_x", alpha=0.2)
axs[0, 1].plot(acc_ref[:, 1], label="acc_y", alpha=0.2)
axs[0, 2].plot(acc_ref[:, 2], label="acc_z", alpha=0.2)
axs[1, 0].plot(omega_ref[:, 0], label="ome_x", alpha=0.2)
axs[1, 1].plot(omega_ref[:, 1], label="ome_y", alpha=0.2)
axs[1, 2].plot(omega_ref[:, 2], label="ome_z", alpha=0.2)

axs[0, 0].plot(filt_aref[:, 0], label="filt_acc_x")
axs[0, 1].plot(filt_aref[:, 1], label="filt_acc_y")
axs[0, 2].plot(filt_aref[:, 2], label="filt_acc_z")
axs[1, 0].plot(filt_oref[:, 0], label="filt_ome_x")
axs[1, 1].plot(filt_oref[:, 1], label="filt_ome_y")
axs[1, 2].plot(filt_oref[:, 2], label="filt_ome_z")

axs[0, 0].plot(savgol_aref[:, 0], label="savgol_acc_x")
axs[0, 1].plot(savgol_aref[:, 1], label="savgol_acc_y")
axs[0, 2].plot(savgol_aref[:, 2], label="savgol_acc_z")
axs[1, 0].plot(savgol_oref[:, 0], label="savgol_ome_x")
axs[1, 1].plot(savgol_oref[:, 1], label="savgol_ome_y")
axs[1, 2].plot(savgol_oref[:, 2], label="savgol_ome_z")

for ax in axs.flatten():
    ax.grid()
    ax.legend()
# axs[0, 0].set_xlim(800, 3000)
# axs[0, 0].set_ylim(-0.5, 1.2)
plt.show()

In [ ]:
assert False

In [ ]:
# og_file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-standard-road-v2.csv"
# save_file_path = "/Users/jozbee/work/eng/comp/data/sms_00_sms_drive.csv"

# df = pd.read_csv(og_file_path)
# df = df[: sms_aref.shape[0]]

# ks = df.keys()
# for i in range(3):
#     df[ks[i + 1]] = sms_aref[:, i]
#     df[ks[i + 1 + 3]] = sms_oref[:, i]

# df[ks[0]] = np.arange(sms_aref.shape[0], dtype=float) * 0.005

# df.to_csv(save_file_path, index=False)

In [ ]:
# og_file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-standard-road-v2.csv"
# save_file_path = "/Users/jozbee/work/eng/comp/data/sms_specific-forces-standard-road-v2.csv"

# df = pd.read_csv(og_file_path)

# ks = df.keys()

# ts = np.array(df[ks[0]])
# diff = np.abs(np.diff(ts))
# avg_diff = np.mean(diff)
# std_diff = np.std(diff)
# bad_indices = np.where(diff > avg_diff + std_diff)[0] + 1 # off by one
# start_index = bad_indices[-2] + 5 * 200
# end_index = bad_indices[-1] - 1

# ts = ts[start_index: end_index + 1]
# df = df[start_index: end_index + 1]

# for i in range(3):
#     df[ks[i + 1]] = sms_aref[:, i]
#     df[ks[i + 1 + 3]] = sms_oref[:, i]

# df.to_csv(save_file_path, index=False)